In [347]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor

import sklearn
sklearn.set_config(transform_output="pandas")
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import HuberRegressor, Ridge, Lasso, RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.ensemble import StackingRegressor, VotingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, OrdinalEncoder, TargetEncoder, FunctionTransformer

In [348]:
df = pd.read_csv('../train.csv')
df=df.drop_duplicates()
df = df.drop('Id', axis=1, errors='ignore')
condition = (df['GrLivArea'] > 4000) & (df['SalePrice'] < 160001)
df=df[~condition]

Q1_price = df['SalePrice'].quantile(0.25)
Q3_price = df['SalePrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound_price = Q1_price - 1.5 * IQR_price
upper_bound_price = Q3_price + 1.5 * IQR_price
    
condition_price = (df['SalePrice'] < lower_bound_price) | (df['SalePrice'] > upper_bound_price)
df = df[~condition_price]

Q1_lot = df['LotArea'].quantile(0.25)
Q3_lot = df['LotArea'].quantile(0.75)
IQR_lot = Q3_lot - Q1_lot
upper_bound_lot = Q3_lot + 3 * IQR_lot

condition_lot = df['LotArea'] > upper_bound_lot
df = df[~condition_lot]


cond_frontage = df['LotFrontage'] > 250
df = df[~cond_frontage]

df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                          df['3SsnPorch'] + df['ScreenPorch'])
cond_porch = df['TotalPorchSF'] > 700
df = df[~cond_porch]
df = df.drop('TotalPorchSF', axis=1)
    

In [349]:
X, y = df.drop('SalePrice', axis=1), df['SalePrice']

y_log = np.log1p(y)

X_train, X_valid, y_train_log, y_valid_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

In [350]:
cat_features = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
                'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
                'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
                'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish',
                'PavedDrive', 'Fence', 'MiscFeature','SaleType', 'SaleCondition', 'PoolQC', 'ExterQual','ExterCond',
                'BsmtQual', 'BsmtCond','HeatingQC','KitchenQual','FireplaceQu', 'GarageQual', 'GarageCond']
num_features = ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
                'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
                'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
                'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
                'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
                'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea', 'MoSold', 'YrSold', '3SsnPorch', 'MiscVal' ]

In [ ]:
def clean_data(X):
    X=X.copy()
    X['MasVnrType'] = X['MasVnrType'].fillna('None')
    X['FireplaceQu'] = X['FireplaceQu'].fillna('None')
    X['GarageQual'] = X['GarageQual'].fillna('None')
    X['GarageFinish'] = X['GarageFinish'].fillna('None')
    X['GarageType'] = X['GarageType'].fillna('None')
    X['GarageCond'] = X['GarageCond'].fillna('None')

    X['Alley'] = X['Alley'].fillna('None')
    X['PoolQC'] = X['PoolQC'].fillna('None')
    X['GarageType'] = X['GarageType'].fillna('None') 

    X['GarageArea'] = X['GarageArea'].fillna(0) 
    X['MasVnrArea'] = X['MasVnrArea'].fillna(0)

    X['GarageYrBlt'] = X['GarageYrBlt'].fillna(X['YearBuilt'])
    X['GarageAge'] = X['YrSold'] - X['GarageYrBlt']
    X['RemAge'] = X['YrSold'] - X['YearRemodAdd']
    X['HouseAge'] = X['YrSold'] - X['YearBuilt']
    X['IsNew'] = (X['HouseAge'] <= 5).astype(int)
    X['IsOld'] = (X['HouseAge'] >= 70).astype(int)
    X['IsHistoric'] = (X['HouseAge'] >= 100).astype(int)
    X['TotalBath'] = (X['FullBath'] + (0.5 * X['HalfBath']) +  #добавлено
                      X['BsmtFullBath'] + (0.5 * X['BsmtHalfBath']))
    X['HasPool'] = X['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
    X['TotalSF'] = (X['GrLivArea'] + 
                    X['TotalBsmtSF'].fillna(0) + 
                    X['GarageArea'].fillna(0) +
                    X['WoodDeckSF'] + 
                    X['OpenPorchSF'] + 
                    X['EnclosedPorch'] + 
                    X['3SsnPorch'] + 
                    X['ScreenPorch'])
    
    X['TotalBathrooms'] = X['FullBath'] + 0.5*X['HalfBath'] + X['BsmtFullBath'] + 0.5*X['BsmtHalfBath']
    X['TotalRooms'] = X['TotRmsAbvGrd'] + X['BedroomAbvGr']
    X['AreaPerRoom'] = X['GrLivArea'] / (X['TotRmsAbvGrd'] + 1)
    X['QualityScore'] = X['OverallQual'] * X['OverallCond']
    X['IsRenovated'] = (X['YearRemodAdd'] != X['YearBuilt']).astype(int)
    X['YearsSinceRenovation'] = X['YrSold'] - X['YearRemodAdd']
    X['YearBuilt_Qual'] = (X["YearBuilt"] * X["OverallQual"])
    X['TotalSF_Qual'] = (X["TotalSF"] * X["OverallQual"])
    return X

new_num_features = num_features + ['GarageAge', 'RemAge', 'HouseAge', 'TotalSF', 'TotalBath', 
                                   'IsNew', 'IsOld', 'IsHistoric', 'HasPool',
                                   'TotalBathrooms', 'TotalRooms', 'AreaPerRoom', 
                                   'QualityScore', 'IsRenovated', 'YearsSinceRenovation']

def imputer_groupby_Neighborhood(X):
    X = X.copy()
    X['LotFrontage'] = X.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
    return X

cleaner = FunctionTransformer(clean_data)
group_imputer = FunctionTransformer(imputer_groupby_Neighborhood)

drop_features = ['MiscFeature', 'Utilities', 'PoolQC', 'Alley', 'Fence',
                 'LowQualFinSF', 'MiscVal', 'HalfBath', 'BsmtHalfBath', 'Street', 'LandSlope'] 

In [352]:
binary_and_low_cardinality = []   
high_cardinality = []       

for col in cat_features:
    n_unique = X_train[col].nunique()
    if n_unique <= 10:
        binary_and_low_cardinality.append(col)
    else:
        high_cardinality.append(col)

my_imputer = ColumnTransformer(
    transformers = [
        ('drop_features', 'drop', drop_features),
        
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), new_num_features),
        
        # ('cat', Pipeline([
        #     ('imputer', SimpleImputer(strategy='most_frequent')),
        #     ('encoder', TargetEncoder(smooth=10, target_type='continuous'))
        # ]), cat_features)
        
        # Бинарные - OneHot
        ('binary', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), binary_and_low_cardinality),
        
        # Высокая кардинальность - FrequencyEncoder
        ('high_cat_freq', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', TargetEncoder(smooth=10))
        ]), high_cardinality)
    ],
    verbose_feature_names_out=False,
    remainder='drop'
)

preprocessor = Pipeline(
    [
        ('cleaner', cleaner),
        ('group_imputer', group_imputer),
        ('imputer', my_imputer)
    ]
)

In [353]:
preprocessor.fit(X_train, y_train_log)

X_train_processed = preprocessor.transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

In [354]:
# X_train_array = np.array(X_train_processed, dtype=np.float64)
# X_valid_array = np.array(X_valid_processed, dtype=np.float64)


# scaler_lasso = StandardScaler()
# X_train_scaled = scaler_lasso.fit_transform(X_train_array)
# X_valid_scaled = scaler_lasso.transform(X_valid_array)

# # ПРИНУДИТЕЛЬНО КОНВЕРТИРУЕМ В NUMPY
# X_train_scaled = np.asarray(X_train_scaled)
# X_valid_scaled = np.asarray(X_valid_scaled)

# print(f"X_train_scaled тип: {type(X_train_scaled)}")
# print(f"X_train_scaled форма: {X_train_scaled.shape}")

# # Проверяем на бесконечности (теперь должно работать)
# if np.isinf(X_train_scaled).any():
#     X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
#     X_valid_scaled = np.nan_to_num(X_valid_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# alpha = 0.005

# lasso = Lasso(alpha=alpha, random_state=42, max_iter=10000)
# lasso.fit(X_train_scaled, y_train_log.values)

# selected_mask = np.abs(lasso.coef_) > 1e-5
# n_selected = np.sum(selected_mask)
# print(f"Alpha = {alpha}")
# print(f"Отобрано признаков: {n_selected} из {X_train_scaled.shape[1]}")

In [355]:
X_train_array = np.array(X_train_processed, dtype=np.float64)
X_valid_array = np.array(X_valid_processed, dtype=np.float64)


scaler_lasso = StandardScaler()
X_train_scaled = scaler_lasso.fit_transform(X_train_array)
X_valid_scaled = scaler_lasso.transform(X_valid_array)

# ПРИНУДИТЕЛЬНО КОНВЕРТИРУЕМ В NUMPY
X_train_scaled = np.asarray(X_train_scaled)
X_valid_scaled = np.asarray(X_valid_scaled)

print(f"X_train_scaled тип: {type(X_train_scaled)}")
print(f"X_train_scaled форма: {X_train_scaled.shape}")

# Проверяем на бесконечности (теперь должно работать)
if np.isinf(X_train_scaled).any():
    X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    X_valid_scaled = np.nan_to_num(X_valid_scaled, nan=0.0, posinf=0.0, neginf=0.0)

alpha = 0.004

lasso = Lasso(alpha=alpha, random_state=42, max_iter=10000)
lasso.fit(X_train_scaled, y_train_log.values)

selected_mask = np.abs(lasso.coef_) > 1e-5
n_selected = np.sum(selected_mask)
print(f"Alpha = {alpha}")
print(f"Отобрано признаков: {n_selected} из {X_train_scaled.shape[1]}")

X_train_scaled тип: <class 'numpy.ndarray'>
X_train_scaled форма: (1092, 249)
Alpha = 0.004
Отобрано признаков: 76 из 249


In [356]:
# # ============================================
# # ПОДБОР ОПТИМАЛЬНОГО ALPHA ДЛЯ LASSO
# # ============================================
# print("="*60)
# print("ПОДБОР ОПТИМАЛЬНОГО ALPHA ДЛЯ LASSO")
# print("="*60)

# # Подготавливаем данные (уже есть X_train_processed и X_valid_processed)
# X_train_array = np.array(X_train_processed, dtype=np.float64)
# X_valid_array = np.array(X_valid_processed, dtype=np.float64)

# # Стандартизация
# scaler_lasso = StandardScaler()
# X_train_scaled = scaler_lasso.fit_transform(X_train_array)
# X_valid_scaled = scaler_lasso.transform(X_valid_array)

# # Принудительно конвертируем в numpy
# X_train_scaled = np.asarray(X_train_scaled)
# X_valid_scaled = np.asarray(X_valid_scaled)

# print(f"X_train_scaled форма: {X_train_scaled.shape}")

# # Проверяем на бесконечности
# if np.isinf(X_train_scaled).any():
#     print("⚠️ Найдены бесконечности, заменяем...")
#     X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
#     X_valid_scaled = np.nan_to_num(X_valid_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# # Разные alpha для тестирования
# alphas = [0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.008, 0.01, 0.015, 0.02]

# results = []

# print("\nТестирование alpha:")
# print("-" * 70)

# for alpha in alphas:
#     # Обучаем Lasso
#     lasso = Lasso(alpha=alpha, random_state=42, max_iter=10000)
#     lasso.fit(X_train_scaled, y_train_log.values)
    
#     # Отбираем признаки
#     selected_mask = np.abs(lasso.coef_) > 1e-5
#     n_selected = np.sum(selected_mask)
    
#     print(f"alpha={alpha:.4f}: {n_selected} признаков", end="")
    
#     # Если признаков слишком мало, пропускаем
#     if n_selected < 10:
#         print(" (слишком мало, пропускаем)")
#         continue
    
#     # Отбираем данные
#     X_train_selected = X_train_scaled[:, selected_mask]
#     X_valid_selected = X_valid_scaled[:, selected_mask]
    
#     # Быстрое обучение CatBoost
#     cb_temp = CatBoostRegressor(
#         bootstrap_type='Bayesian',
#         iterations=500,
#         learning_rate=0.0279,
#         depth=3,
#         l2_leaf_reg=12.1,
#         bagging_temperature=5.06,
#         random_strength=7.64,
#         colsample_bylevel=0.49,
#         min_data_in_leaf=225,
#         eval_metric='RMSE',
#         loss_function='RMSE',
#         early_stopping_rounds=30,
#         max_bin=128,
#         verbose=False,
#         random_seed=42
#     )
    
#     cb_temp.fit(
#         X_train_selected, 
#         y_train_log.values, 
#         eval_set=(X_valid_selected, y_valid_log.values),
#         verbose=False
#     )
    
#     # Оценка
#     y_pred_valid = cb_temp.predict(X_valid_selected)
#     valid_rmsle = np.sqrt(mean_squared_log_error(
#         np.expm1(y_valid_log.values), 
#         np.expm1(y_pred_valid)
#     ))
    
#     y_pred_train = cb_temp.predict(X_train_selected)
#     train_rmsle = np.sqrt(mean_squared_log_error(
#         np.expm1(y_train_log.values), 
#         np.expm1(y_pred_train)
#     ))
    
#     diff = (valid_rmsle - train_rmsle) / train_rmsle * 100
    
#     print(f", Valid RMSLE={valid_rmsle:.6f}, Разница={diff:.2f}%")
    
#     results.append({
#         'alpha': alpha,
#         'n_features': n_selected,
#         'train_rmsle': train_rmsle,
#         'valid_rmsle': valid_rmsle,
#         'diff': diff
#     })

# # ============================================
# # ВЫВОД РЕЗУЛЬТАТОВ
# # ============================================
# print("\n" + "="*70)
# print("РЕЗУЛЬТАТЫ ПОДБОРА ALPHA")
# print("="*70)

# if results:
#     results_df = pd.DataFrame(results)
#     print(results_df[['alpha', 'n_features', 'valid_rmsle', 'diff']].to_string(index=False))
    
#     # Находим лучший alpha
#     best_result = results_df.loc[results_df['valid_rmsle'].idxmin()]
    
#     print("\n" + "="*70)
#     print(f"🏆 ЛУЧШИЙ ALPHA: {best_result['alpha']:.4f}")
#     print("="*70)
#     print(f"Valid RMSLE: {best_result['valid_rmsle']:.6f}")
#     print(f"Train RMSLE: {best_result['train_rmsle']:.6f}")
#     print(f"Разница: {best_result['diff']:.2f}%")
#     print(f"Отобрано признаков: {int(best_result['n_features'])}")
    
#     # ============================================
#     # ИСПОЛЬЗУЕМ ЛУЧШИЙ ALPHA
#     # ============================================
#     best_alpha = best_result['alpha']
    
#     # Обучаем Lasso с лучшим alpha
#     lasso_best = Lasso(alpha=best_alpha, random_state=42, max_iter=10000)
#     lasso_best.fit(X_train_scaled, y_train_log.values)
    
#     selected_mask_best = np.abs(lasso_best.coef_) > 1e-5
#     n_selected_best = np.sum(selected_mask_best)
    
#     print(f"\n✅ Используем alpha = {best_alpha}")
#     print(f"Отобрано признаков: {n_selected_best} из {X_train_scaled.shape[1]}")
    
#     X_train_selected_best = X_train_scaled[:, selected_mask_best]
#     X_valid_selected_best = X_valid_scaled[:, selected_mask_best]
    
# else:
#     print("⚠️ Нет подходящих результатов, используем alpha = 0.005")
#     best_alpha = 0.005
#     selected_mask_best = np.abs(lasso.coef_) > 1e-5
#     X_train_selected_best = X_train_scaled[:, selected_mask_best]
#     X_valid_selected_best = X_valid_scaled[:, selected_mask_best]

# # ============================================
# # ОБУЧЕНИЕ CATBOOST С ЛУЧШИМ ALPHA
# # ============================================
# print("\n" + "="*70)
# print("ОБУЧЕНИЕ CATBOOST С ЛУЧШИМ ALPHA")
# print("="*70)


In [357]:
X_train_selected = X_train_scaled[:, selected_mask]
X_valid_selected = X_valid_scaled[:, selected_mask]

# cb = CatBoostRegressor(
    # iterations=550,
    # learning_rate=0.299266706369678,
    # depth=3,
    # l2_leaf_reg=10.655197226966266,
    # bagging_temperature=1.7457235809550053,
    # random_strength=0.7507556902584652,
    # subsample=0.8352580408542853,
    # colsample_bylevel=0.9043302098880885,
    # min_data_in_leaf=40,
    # eval_metric='RMSE',
    # early_stopping_rounds=50,
    # verbose=100,
    # random_seed=42
# )

# cb.fit(
#     X_train_selected,
#     y_train_log.values,
#     eval_set=(X_valid_selected, y_valid_log),
#     plot=True
# )


In [358]:
# base_models = [
#     ('cb', CatBoostRegressor(
#     bootstrap_type = 'Bayesian',
#     iterations=650,
#     learning_rate=0.20024533573359862,
#     depth=4,
#     l2_leaf_reg=23.806251296453688,
#     bagging_temperature=1.2758228110041578,
#     random_strength=1.084888727839387,
#     colsample_bylevel=0.7190326638991136,
#     min_data_in_leaf=90,
#     eval_metric='RMSE',
#     loss_function = 'RMSE',
#     early_stopping_rounds=50,
#     border_count = 192,
#     verbose=0,
#     random_seed=42
# )),

#     ('lightgbm', LGBMRegressor(
#         n_estimators=1000, 
#         learning_rate=0.04278016409633336, 
#         num_leaves=20,
#         max_depth = 6, 
#         reg_lambda=17.688553398529873,      # L2 регуляризация
#         reg_alpha=0.7640078312183015,        # L1 регуляризация
#         min_child_samples = 30,
#         subsample = 0.8323892441671329,
#         colsample_bytree = 0.6099938256768226,
#         min_split_gain = 1.7525382418403374e-05,
#         boosting_type = 'gbdt',
#         objective = 'regression',
#         metric = 'RMSE',
#         verbose = -1,
#         random_state = 42,
#         n_jobs = -1
#     ))
# ]

# final_model = Ridge()

# stacking_model = StackingRegressor(
#     estimators=base_models,
#     final_estimator=final_model,
#     cv=5 # Кросс-валидация для обучения мета-модели
# )
# stacking_model.fit(X_train_selected, y_train_log)

# y_pred_train_log = stacking_model.predict(X_train_selected)
# y_pred_valid_log = stacking_model.predict(X_valid_selected)

# y_pred_train_orig = np.expm1(y_pred_train_log)
# y_pred_valid_orig = np.expm1(y_pred_valid_log)

# # Оригинальные значения
# y_train_orig = np.expm1(y_train_log.values)
# y_valid_orig = np.expm1(y_valid_log.values)

In [359]:
catboost_model = CatBoostRegressor(
    bootstrap_type='Bayesian',
    iterations=650,
    learning_rate=0.20024533573359862,
    depth=4,
    l2_leaf_reg=23.806251296453688,
    bagging_temperature=1.2758228110041578,
    random_strength=1.084888727839387,
    colsample_bylevel=0.7190326638991136,
    min_data_in_leaf=90,
    eval_metric='RMSE',
    loss_function='RMSE',
    early_stopping_rounds=50,
    border_count=192,
    verbose=100,
    random_seed=42
)

catboost_model.fit(
    X_train_selected, y_train_log,
    eval_set=(X_valid_selected, y_valid_log),
    verbose=50
)

lgb_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.04278016409633336,
    num_leaves=20,
    max_depth=6,
    reg_lambda=17.688553398529873,
    reg_alpha=0.7640078312183015,
    min_child_samples=30,
    subsample=0.8323892441671329,
    colsample_bytree=0.6099938256768226,
    min_split_gain=1.7525382418403374e-05,
    boosting_type='gbdt',
    objective='regression',
    metric='rmse',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train_selected, y_train_log,
    eval_set=[(X_valid_selected, y_valid_log)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

base_models = [
    ('cb', catboost_model),
    ('lightgbm', lgb_model)
]

stacking_model = StackingRegressor(
    estimators=base_models,
    # final_estimator=Ridge(),
    cv='prefit'
)
stacking_model.fit(X_train_selected, y_train_log)

y_pred_train_log = stacking_model.predict(X_train_selected)
y_pred_valid_log = stacking_model.predict(X_valid_selected)

y_pred_train_orig = np.expm1(y_pred_train_log)
y_pred_valid_orig = np.expm1(y_pred_valid_log)

# Оригинальные значения
y_train_orig = np.expm1(y_train_log.values)
y_valid_orig = np.expm1(y_valid_log.values)

0:	learn: 0.3154475	test: 0.3281855	best: 0.3281855 (0)	total: 3.77ms	remaining: 2.45s
50:	learn: 0.1088868	test: 0.1216978	best: 0.1216978 (50)	total: 124ms	remaining: 1.46s
100:	learn: 0.0955118	test: 0.1138131	best: 0.1138131 (100)	total: 211ms	remaining: 1.15s
150:	learn: 0.0881960	test: 0.1101851	best: 0.1101851 (150)	total: 323ms	remaining: 1.07s
200:	learn: 0.0822446	test: 0.1080438	best: 0.1080438 (200)	total: 395ms	remaining: 882ms
250:	learn: 0.0773149	test: 0.1072330	best: 0.1068022 (222)	total: 528ms	remaining: 840ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.1068021576
bestIteration = 222

Shrink model to first 223 iterations.
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 0.156211
[100]	valid_0's rmse: 0.125471
[150]	valid_0's rmse: 0.1178
[200]	valid_0's rmse: 0.114773
[250]	valid_0's rmse: 0.11335
[300]	valid_0's rmse: 0.112582
[350]	valid_0's rmse: 0.11193
[400]	valid_0's rmse: 0.111526
[450]	valid_0's rmse: 0

In [360]:
train_rmse = np.sqrt(mean_squared_error(y_train_orig, y_pred_train_orig))
train_mae = mean_absolute_error(y_train_orig, y_pred_train_orig)
train_r2 = r2_score(y_train_orig, y_pred_train_orig)

valid_rmse = np.sqrt(mean_squared_error(y_valid_orig, y_pred_valid_orig))
valid_mae = mean_absolute_error(y_valid_orig, y_pred_valid_orig)
valid_r2 = r2_score(y_valid_orig, y_pred_valid_orig)

# RMSLE (главная метрика соревнования)
train_rmsle = np.sqrt(mean_squared_log_error(y_train_orig, y_pred_train_orig))
valid_rmsle = np.sqrt(mean_squared_log_error(y_valid_orig, y_pred_valid_orig))

print("Train Metrics:")
print(f"RMSE: {train_rmse:.4f}")
print(f"MAE: {train_mae:.4f}")
print(f"R²: {train_r2:.4f}")
print("\nValidation Metrics:")
print(f"RMSE: {valid_rmse:.4f}")
print(f"MAE: {valid_mae:.4f}")
print(f"R²: {valid_r2:.4f}")

Train Metrics:
RMSE: 11763.1210
MAE: 8433.7918
R²: 0.9601

Validation Metrics:
RMSE: 15616.8087
MAE: 10784.7127
R²: 0.9305


In [361]:
y_pred_train_log = stacking_model.predict(X_train_selected)
y_pred_valid_log = stacking_model.predict(X_valid_selected)

# Метрики на логарифмической шкале (для RMSLE нужно expm1)
train_rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_train_log.values), 
    np.expm1(y_pred_train_log)
))
valid_rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_valid_log.values), 
    np.expm1(y_pred_valid_log)
))

# Дополнительные метрики для понимания
train_rmse_orig = np.sqrt(mean_squared_error(
    np.expm1(y_train_log.values), 
    np.expm1(y_pred_train_log)
))
valid_rmse_orig = np.sqrt(mean_squared_error(
    np.expm1(y_valid_log.values), 
    np.expm1(y_pred_valid_log)
))

print("="*60)
print("МЕТРИКИ ДЛЯ СОРЕВНОВАНИЯ")
print("="*60)
print(f"Train RMSLE: {train_rmsle:.6f}")
print(f"Valid RMSLE: {valid_rmsle:.6f}")
print(f"Разница RMSLE: {((valid_rmsle - train_rmsle) / train_rmsle * 100):.2f}%")
print(f"\nRMSE (ориг. шкала):")
print(f"Train: ${train_rmse_orig:.2f}")
print(f"Valid: ${valid_rmse_orig:.2f}")

МЕТРИКИ ДЛЯ СОРЕВНОВАНИЯ
Train RMSLE: 0.073844
Valid RMSLE: 0.107216
Разница RMSLE: 45.19%

RMSE (ориг. шкала):
Train: $11763.12
Valid: $15616.81


In [362]:
# import optuna
# import numpy as np
# import pandas as pd
# from catboost import CatBoostRegressor
# from lightgbm import LGBMRegressor
# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_squared_error, mean_squared_log_error
# import warnings
# warnings.filterwarnings('ignore')

# # ============================================================
# # ОПТИМИЗАЦИЯ CATBOOST (с кросс-валидацией)
# # ============================================================

# def objective_catboost(trial, X, y, n_folds=5):
#     """
#     Оптимизация CatBoost с кросс-валидацией
#     """
#     params = {
#         # Основные параметры (уменьшаем сложность)
#         'iterations': trial.suggest_int('iterations', 300, 700, step=50),
#         'learning_rate': trial.suggest_float('learning_rate', 0.2, 0.38, log=True),
#         'depth': trial.suggest_int('depth', 2, 4),
        
#         # Усиленная регуляризация
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 8, 30, log=True),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 40, 120, step=10),
        
#         # Контроль переобучения
#         'bagging_temperature': trial.suggest_float('bagging_temperature', 1.0, 5.0),
#         'random_strength': trial.suggest_float('random_strength', 1.0, 4.0, log=True),
#         'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.7, 0.9),
        
#         # Дополнительные
#         'border_count': trial.suggest_int('border_count', 64, 192, step=32),
        
#         # Фиксированные
#         'bootstrap_type': 'Bayesian',
#         'eval_metric': 'RMSE',
#         'loss_function': 'RMSE',
#         'early_stopping_rounds': 50,
#         'verbose': 0,
#         'random_seed': 42,
#     }
    
#     # Кросс-валидация
#     cv_scores = []
#     kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
#     for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#         X_tr, X_val = X[train_idx], X[val_idx]
#         y_tr, y_val = y[train_idx], y[val_idx]
        
#         model = CatBoostRegressor(**params)
#         model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
        
#         y_pred = model.predict(X_val)
#         rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(y_pred)))
#         cv_scores.append(rmsle)
        
#         trial.report(np.mean(cv_scores), fold)
#         if trial.should_prune():
#             raise optuna.TrialPruned()
    
#     return np.mean(cv_scores)


# # ============================================================
# # ОПТИМИЗАЦИЯ LIGHTGBM (с кросс-валидацией)
# # ============================================================

# def objective_lightgbm(trial, X, y, n_folds=5):
#     """
#     Оптимизация LightGBM с кросс-валидацией
#     """
#     params = {
#         # Основные параметры
#         'n_estimators': trial.suggest_int('n_estimators', 500, 1200, step=100),
#         'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
#         'num_leaves': trial.suggest_int('num_leaves', 20, 50),
#         'max_depth': trial.suggest_int('max_depth', 3, 6),
        
#         # Усиленная регуляризация
#         'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 30, log=True),
#         'reg_alpha': trial.suggest_float('reg_alpha', 0.5, 15, log=True),
#         'min_child_samples': trial.suggest_int('min_child_samples', 30, 120, step=10),
#         'subsample': trial.suggest_float('subsample', 0.6, 0.85),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.85),
        
#         # Дополнительные
#         'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
        
#         # Фиксированные
#         'boosting_type': 'gbdt',
#         'objective': 'regression',
#         'metric': 'rmse',
#         'verbose': -1,
#         'random_state': 42,
#         'n_jobs': -1,
#     }
    
#     # Кросс-валидация
#     import lightgbm as lgb
#     cv_scores = []
#     kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
#     for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#         X_tr, X_val = X[train_idx], X[val_idx]
#         y_tr, y_val = y[train_idx], y[val_idx]
        
#         model = LGBMRegressor(**params)
#         model.fit(
#             X_tr, y_tr,
#             eval_set=[(X_val, y_val)],
#             eval_metric='rmse',
#             callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
#         )
        
#         y_pred = model.predict(X_val)
#         rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(y_pred)))
#         cv_scores.append(rmsle)
        
#         trial.report(np.mean(cv_scores), fold)
#         if trial.should_prune():
#             raise optuna.TrialPruned()
    
#     return np.mean(cv_scores)


# # ============================================================
# # ЗАПУСК ОПТИМИЗАЦИИ
# # ============================================================

# print("="*80)
# print("ОПТИМИЗАЦИЯ CATBOOST И LIGHTGBM")
# print("Фокус: уменьшение переобучения и минимальный RMSLE")
# print("Кросс-валидация: 5-fold")
# print("="*80)

# # Проверка данных
# print(f"\nДанные для оптимизации:")
# print(f"  X_train_selected: {X_train_selected.shape}")
# print(f"  X_valid_selected: {X_valid_selected.shape}")
# print(f"  y_train: {len(y_train_log)}")
# print(f"  y_valid: {len(y_valid_log)}")

# # Преобразуем в numpy если нужно
# X_train_opt = np.array(X_train_selected)
# X_valid_opt = np.array(X_valid_selected)
# y_train_opt = y_train_log.values if hasattr(y_train_log, 'values') else y_train_log
# y_valid_opt = y_valid_log.values if hasattr(y_valid_log, 'values') else y_valid_log

# # Объединяем данные для кросс-валидации
# X_combined = np.vstack([X_train_opt, X_valid_opt])
# y_combined = np.hstack([y_train_opt, y_valid_opt])

# print(f"\nОбъединенные данные для CV:")
# print(f"  X_combined: {X_combined.shape}")
# print(f"  y_combined: {y_combined.shape}")

# print(f"\nТипы данных после преобразования:")
# print(f"  X_combined: {type(X_combined)}, {X_combined.dtype}")
# print(f"  y_combined: {type(y_combined)}")

# # ============================================================
# # ОПТИМИЗАЦИЯ CATBOOST (5-fold CV)
# # ============================================================

# print("\n" + "="*80)
# print("ОПТИМИЗАЦИЯ CATBOOST (5-fold CV)")
# print("="*80)

# study_cb = optuna.create_study(
#     direction='minimize',
#     sampler=optuna.samplers.TPESampler(seed=42),
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
# )

# study_cb.optimize(
#     lambda trial: objective_catboost(
#         trial, X_combined, y_combined, n_folds=5
#     ),
#     n_trials=200,
#     show_progress_bar=True
# )

# print("\n" + "="*80)
# print("ЛУЧШИЕ ПАРАМЕТРЫ CATBOOST (5-fold CV)")
# print("="*80)
# print(f"Средний RMSLE (CV): {study_cb.best_value:.6f}")
# print("\nПараметры:")
# for key, value in study_cb.best_params.items():
#     print(f"  {key}: {value}")

# # Сохраняем лучшие параметры CatBoost
# cb_best_params = study_cb.best_params.copy()
# cb_best_params.update({
#     'bootstrap_type': 'Bayesian',
#     'eval_metric': 'RMSE',
#     'loss_function': 'RMSE',
#     'early_stopping_rounds': 50,
#     'random_seed': 42
# })

# # ============================================================
# # ОПТИМИЗАЦИЯ LIGHTGBM (5-fold CV)
# # ============================================================

# print("\n" + "="*80)
# print("ОПТИМИЗАЦИЯ LIGHTGBM (5-fold CV)")
# print("="*80)

# import lightgbm as lgb

# study_lgb = optuna.create_study(
#     direction='minimize',
#     sampler=optuna.samplers.TPESampler(seed=42),
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
# )

# study_lgb.optimize(
#     lambda trial: objective_lightgbm(
#         trial, X_combined, y_combined, n_folds=5
#     ),
#     n_trials=200,
#     show_progress_bar=True
# )

# print("\n" + "="*80)
# print("ЛУЧШИЕ ПАРАМЕТРЫ LIGHTGBM (5-fold CV)")
# print("="*80)
# print(f"Средний RMSLE (CV): {study_lgb.best_value:.6f}")
# print("\nПараметры:")
# for key, value in study_lgb.best_params.items():
#     print(f"  {key}: {value}")

# # Сохраняем лучшие параметры LightGBM
# lgb_best_params = study_lgb.best_params.copy()
# lgb_best_params.update({
#     'boosting_type': 'gbdt',
#     'objective': 'regression',
#     'metric': 'rmse',
#     'verbose': -1,
#     'random_state': 42,
#     'n_jobs': -1
# })

# # ============================================================
# # ВЫВОД ГОТОВЫХ ПАРАМЕТРОВ
# # ============================================================

# print("\n" + "="*80)
# print("ГОТОВЫЕ ПАРАМЕТРЫ ДЛЯ КОПИРОВАНИЯ")
# print("="*80)

# print("\n# CatBoost лучшие параметры (5-fold CV):")
# print("cb_best_params = {")
# for key, value in cb_best_params.items():
#     if isinstance(value, float):
#         print(f"    '{key}': {value},")
#     else:
#         print(f"    '{key}': {value},")
# print("}")

# print("\n# LightGBM лучшие параметры (5-fold CV):")
# print("lgb_best_params = {")
# for key, value in lgb_best_params.items():
#     if isinstance(value, float):
#         print(f"    '{key}': {value},")
#     else:
#         print(f"    '{key}': {value},")
# print("}")

# print("\n" + "="*80)
# print("ОПТИМИЗАЦИЯ ЗАВЕРШЕНА")
# print("="*80)
# print(f"\nCatBoost CV RMSLE: {study_cb.best_value:.6f}")
# print(f"LightGBM CV RMSLE: {study_lgb.best_value:.6f}")

In [ ]:
# final_pipeline = Pipeline([('preprocessor', preprocessor), ('CatBoost', cb)])
# final_pipeline.fit(X_train, y_train_log)
# test = pd.read_csv('../test.csv')
# result = np.expm1(final_pipeline.predict(test))
# df_result = pd.DataFrame(test['Id'])
# df_result['SalePrice'] = result
# df_result.to_csv('submission.csv', index=False)

0:	learn: 0.3479244	total: 3.18ms	remaining: 2.86s
100:	learn: 0.1616107	total: 500ms	remaining: 3.96s
200:	learn: 0.1289874	total: 709ms	remaining: 2.46s
300:	learn: 0.1179316	total: 942ms	remaining: 1.87s
400:	learn: 0.1121649	total: 1.11s	remaining: 1.38s
500:	learn: 0.1083098	total: 1.32s	remaining: 1.05s
600:	learn: 0.1052130	total: 1.48s	remaining: 738ms
700:	learn: 0.1027255	total: 1.64s	remaining: 466ms
800:	learn: 0.1006044	total: 1.8s	remaining: 223ms
899:	learn: 0.0986253	total: 1.99s	remaining: 0us


In [ ]:
# import joblib
# joblib.dump(final_pipeline, '../final_pipeline.pkl')

['../final_pipeline.pkl']